# 601 — Explainable Predictive Modeling

## Objective

Evaluate whether the three frozen Phase 4 consensus transcriptomic program scores provide reproducible predictive information about GDSC `LN_IC50` beyond a transparent lineage-only baseline under the prospectively frozen notebook-601 evaluation design.

This notebook treats predictive performance as internal, developmental evidence about resistance-like pharmacogenomic contexts. It does not establish clinical drug resistance prediction, therapeutic efficacy, causal mechanisms, validated biomarkers, or generalization to patients.

## Analytical boundary

Notebook 601 is restricted to the 281 GDSC drugs declared eligible in notebook 600 under the frozen `20 / 3 / 100` lineage-support rule.

For each eligible drug, the primary evaluation compares:

* a lineage-only baseline: `LN_IC50 ~ C(OncotreeLineage)`; and
* a fixed program model: `LN_IC50 ~ C(OncotreeLineage) + CONSENSUS_TX_01 + CONSENSUS_TX_02 + CONSENSUS_TX_03`.

Primary predictive validity is evaluated using `5-fold lineage-stratified cross-validation × 5 repeats`, with deterministic seed `601` and identical held-out partitions for the baseline and program models.

The primary estimand is prediction for a previously unseen cell-line model belonging to a lineage already represented for the same known drug. Completely unseen-lineage transportability is evaluated separately through the prospectively defined leave-one-supported-lineage-out stress test.

No model family, feature subset, drug subset, threshold, preprocessing rule, seed, or resampling structure will be selected according to observed predictive performance.

## Predictive-validity criterion

A drug is eligible for subsequent primary program-level SHAP interpretation in notebook 602 only if all prospectively frozen criteria are satisfied:

1. `median OOF R²_program ≥ 0.05`;
2. `median ΔR² ≥ 0.02`, where `ΔR² = R²_program − R²_lineage`; and
3. `ΔR² > 0` in at least 4 of the 5 repeated partitions.

Failure to satisfy these criteria is retained as a valid notebook-601 result and does not trigger alternative model fitting, threshold relaxation, feature expansion, or compound rescue.

## Evidence isolation

* **GDSC:** developmental/internal resource used for notebook-601 model fitting and internal predictive evaluation.
* **CTRP and PRISM:** remain sealed external cross-screen resources and are not inspected for predictive outcomes during notebook 601.
* **Phase 5:** functional-vulnerability results are not used for feature selection, drug selection, model selection, threshold definition, or result rescue.

The frozen notebook-600 association results are not used to select drugs or programs for predictive modeling. All three consensus programs enter every primary model jointly.

Residual proliferation and other unresolved cell-line confounding remain explicit limitations rather than grounds for post hoc covariate construction.

Predictive performance and subsequent model attribution describe computational behavior under the frozen evaluation design. They do not establish causality, biological mechanism, therapeutic efficacy, or clinical predictiveness.


In [1]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Resolve frozen notebook-600 handoffs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase6.600.gdsc_analysis_universe",
    "phase6.600.drug_eligibility",
)

input_paths = {
    artifact_id: resolve_artifact_path(artifact_registry, artifact_id)
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(f"{artifact_id}: {project_relative_path(path)}")

phase6.600.gdsc_analysis_universe: data/processed/pharmacogenomic_contexts/600_gdsc_analysis_universe.parquet
phase6.600.drug_eligibility: data/processed/pharmacogenomic_contexts/600_drug_eligibility.csv


In [3]:
# =============================================================================
# Load frozen notebook-600 GDSC handoffs
# =============================================================================

gdsc_universe = pd.read_parquet(
    input_paths["phase6.600.gdsc_analysis_universe"]
)
drug_eligibility = pd.read_csv(
    input_paths["phase6.600.drug_eligibility"]
)

print("GDSC analysis universe shape:", gdsc_universe.shape)
print("Drug eligibility shape:", drug_eligibility.shape)

print("\nGDSC analysis universe columns:")
print(gdsc_universe.columns.tolist())

print("\nDrug eligibility columns:")
print(drug_eligibility.columns.tolist())

print(
    "\nDuplicate ModelID × DRUG_ID rows:",
    gdsc_universe.duplicated(["ModelID", "DRUG_ID"]).sum(),
)

GDSC analysis universe shape: (136176, 7)
Drug eligibility shape: (1037, 6)

GDSC analysis universe columns:
['DRUG_ID', 'ModelID', 'OncotreeLineage', 'response_value', 'response_metric', 'resource', 'evidence_role']

Drug eligibility columns:
['resource_drug_id', 'supported_lineages', 'supported_models', 'eligible', 'resource', 'analysis_universe']

Duplicate ModelID × DRUG_ID rows: 0


In [4]:
# =============================================================================
# Attach frozen consensus-program scores to the GDSC modeling universe
# =============================================================================

program_score_path = resolve_artifact_path(
    artifact_registry,
    "phase6.600.program_score_universe",
)

program_scores = pd.read_parquet(program_score_path)[
    [
        "ModelID",
        "CONSENSUS_TX_01",
        "CONSENSUS_TX_02",
        "CONSENSUS_TX_03",
    ]
]

gdsc_modeling = gdsc_universe.merge(
    program_scores,
    on="ModelID",
    how="left",
    validate="many_to_one",
)

program_columns = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

print("GDSC modeling universe shape:", gdsc_modeling.shape)
print(
    "Rows with missing program scores:",
    gdsc_modeling[program_columns].isna().any(axis=1).sum(),
)

GDSC modeling universe shape: (136176, 10)
Rows with missing program scores: 0


In [5]:
# =============================================================================
# Confirm the frozen primary GDSC drug universe
# =============================================================================

gdsc_eligibility = drug_eligibility.loc[
    (drug_eligibility["resource"].str.upper() == "GDSC")
    & drug_eligibility["eligible"].astype(bool)
].copy()

eligible_drug_ids = set(gdsc_eligibility["resource_drug_id"])
modeling_drug_ids = set(gdsc_modeling["DRUG_ID"])

print("Frozen eligible GDSC drugs:", len(eligible_drug_ids))
print("Drugs represented in modeling universe:", len(modeling_drug_ids))
print("Drug universes identical:", eligible_drug_ids == modeling_drug_ids)

Frozen eligible GDSC drugs: 281
Drugs represented in modeling universe: 281
Drug universes identical: False


In [6]:
# =============================================================================
# Diagnose GDSC drug-identifier representation
# =============================================================================

print("resource_drug_id dtype:", gdsc_eligibility["resource_drug_id"].dtype)
print("DRUG_ID dtype:", gdsc_modeling["DRUG_ID"].dtype)

eligible_drug_ids_str = set(
    gdsc_eligibility["resource_drug_id"].astype(str)
)
modeling_drug_ids_str = set(
    gdsc_modeling["DRUG_ID"].astype(str)
)

print(
    "\nDrug universes identical after string normalization:",
    eligible_drug_ids_str == modeling_drug_ids_str,
)

print(
    "Only in eligibility:",
    sorted(eligible_drug_ids_str - modeling_drug_ids_str)[:10],
)
print(
    "Only in modeling universe:",
    sorted(modeling_drug_ids_str - eligible_drug_ids_str)[:10],
)

resource_drug_id dtype: str
DRUG_ID dtype: int64

Drug universes identical after string normalization: True
Only in eligibility: []
Only in modeling universe: []


In [7]:
# =============================================================================
# Frozen notebook-601 modeling specification
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

TARGET_COLUMN = "response_value"
LINEAGE_COLUMN = "OncotreeLineage"
MODEL_ID_COLUMN = "ModelID"
DRUG_ID_COLUMN = "DRUG_ID"

RANDOM_SEED = 601
N_SPLITS = 5
N_REPEATS = 5

MIN_MEDIAN_R2_PROGRAM = 0.05
MIN_MEDIAN_DELTA_R2 = 0.02
MIN_POSITIVE_DELTA_REPEATS = 4

print("Primary target:", TARGET_COLUMN)
print("Programs:", ", ".join(PROGRAM_COLUMNS))
print(f"Primary CV: {N_SPLITS}-fold × {N_REPEATS} repeats")
print("Frozen seed:", RANDOM_SEED)
print(
    "SHAP eligibility gate:",
    f"median R² >= {MIN_MEDIAN_R2_PROGRAM}, "
    f"median ΔR² >= {MIN_MEDIAN_DELTA_R2}, "
    f"positive ΔR² in >= {MIN_POSITIVE_DELTA_REPEATS}/{N_REPEATS} repeats",
)

Primary target: response_value
Programs: CONSENSUS_TX_01, CONSENSUS_TX_02, CONSENSUS_TX_03
Primary CV: 5-fold × 5 repeats
Frozen seed: 601
SHAP eligibility gate: median R² >= 0.05, median ΔR² >= 0.02, positive ΔR² in >= 4/5 repeats


In [8]:
# =============================================================================
# Establish deterministic row ordering for primary resampling
# =============================================================================

gdsc_modeling = (
    gdsc_modeling
    .sort_values(
        [DRUG_ID_COLUMN, LINEAGE_COLUMN, MODEL_ID_COLUMN],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print("Deterministic modeling rows:", len(gdsc_modeling))

Deterministic modeling rows: 136176


In [9]:
# =============================================================================
# Generate frozen repeated lineage-stratified CV partitions
# =============================================================================

cv_rows = []

for drug_id, drug_df in gdsc_modeling.groupby(DRUG_ID_COLUMN, sort=True):
    splitter = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_SEED,
    )

    for split_idx, (_, test_idx) in enumerate(
        splitter.split(drug_df, drug_df[LINEAGE_COLUMN])
    ):
        repeat = split_idx // N_SPLITS + 1
        fold = split_idx % N_SPLITS + 1

        cv_rows.append(
            pd.DataFrame(
                {
                    DRUG_ID_COLUMN: drug_id,
                    MODEL_ID_COLUMN: drug_df.iloc[test_idx][MODEL_ID_COLUMN].to_numpy(),
                    "repeat": repeat,
                    "fold": fold,
                }
            )
        )

cv_partitions = pd.concat(cv_rows, ignore_index=True)

expected_rows = len(gdsc_modeling) * N_REPEATS

print("CV assignment rows:", len(cv_partitions))
print("Expected rows:", expected_rows)
print(
    "One test assignment per ModelID × drug × repeat:",
    not cv_partitions.duplicated(
        [DRUG_ID_COLUMN, MODEL_ID_COLUMN, "repeat"]
    ).any(),
)
print(
    "All five repeats represented:",
    sorted(cv_partitions["repeat"].unique().tolist()),
)
print(
    "All five folds represented:",
    sorted(cv_partitions["fold"].unique().tolist()),
)

CV assignment rows: 680880
Expected rows: 680880
One test assignment per ModelID × drug × repeat: True
All five repeats represented: [1, 2, 3, 4, 5]
All five folds represented: [1, 2, 3, 4, 5]


In [10]:
# =============================================================================
# Validate lineage representation across primary CV folds
# =============================================================================

partition_lineages = cv_partitions.merge(
    gdsc_modeling[
        [DRUG_ID_COLUMN, MODEL_ID_COLUMN, LINEAGE_COLUMN]
    ],
    on=[DRUG_ID_COLUMN, MODEL_ID_COLUMN],
    how="left",
    validate="many_to_one",
)

lineage_fold_counts = (
    partition_lineages
    .groupby(
        [DRUG_ID_COLUMN, "repeat", LINEAGE_COLUMN],
        observed=True,
    )["fold"]
    .nunique()
)

print(
    "Every drug × repeat × lineage represented in all five folds:",
    lineage_fold_counts.eq(N_SPLITS).all(),
)
print(
    "Minimum folds represented for any drug × repeat × lineage:",
    lineage_fold_counts.min(),
)

Every drug × repeat × lineage represented in all five folds: True
Minimum folds represented for any drug × repeat × lineage: 5


In [11]:
# =============================================================================
# Define frozen primary predictive models
# =============================================================================

def make_lineage_baseline():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "lineage",
                OneHotEncoder(
                    drop="first",
                    handle_unknown="error",
                    sparse_output=False,
                ),
                [LINEAGE_COLUMN],
            ),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LinearRegression()),
        ]
    )


def make_program_model():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "lineage",
                OneHotEncoder(
                    drop="first",
                    handle_unknown="error",
                    sparse_output=False,
                ),
                [LINEAGE_COLUMN],
            ),
            (
                "programs",
                "passthrough",
                PROGRAM_COLUMNS,
            ),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LinearRegression()),
        ]
    )

In [12]:
# =============================================================================
# Define primary fold-level fitting and prediction
# =============================================================================

def fit_predict_primary_fold(
    train_df,
    test_df,
    return_state=False,
):
    baseline_model = make_lineage_baseline()
    program_model = make_program_model()

    baseline_model.fit(
        train_df[[LINEAGE_COLUMN]],
        train_df[TARGET_COLUMN],
    )

    program_model.fit(
        train_df[[LINEAGE_COLUMN, *PROGRAM_COLUMNS]],
        train_df[TARGET_COLUMN],
    )

    predictions = test_df[
        [
            DRUG_ID_COLUMN,
            MODEL_ID_COLUMN,
            LINEAGE_COLUMN,
            TARGET_COLUMN,
            *PROGRAM_COLUMNS,
        ]
    ].copy()

    predictions["prediction_lineage"] = baseline_model.predict(
        test_df[[LINEAGE_COLUMN]]
    )

    predictions["prediction_program"] = program_model.predict(
        test_df[[LINEAGE_COLUMN, *PROGRAM_COLUMNS]]
    )

    if not return_state:
        return predictions

    preprocessor = program_model.named_steps["preprocessor"]
    lineage_encoder = preprocessor.named_transformers_["lineage"]
    linear_model = program_model.named_steps["model"]

    lineage_categories = list(
        lineage_encoder.categories_[0]
    )

    drop_idx = int(lineage_encoder.drop_idx_[0])
    reference_lineage = lineage_categories[drop_idx]

    encoded_lineages = [
        lineage
        for index, lineage in enumerate(lineage_categories)
        if index != drop_idx
    ]

    n_lineage_coefficients = len(encoded_lineages)

    lineage_coefficients = np.asarray(
        linear_model.coef_[:n_lineage_coefficients],
        dtype=float,
    )

    program_coefficients = np.asarray(
        linear_model.coef_[n_lineage_coefficients:],
        dtype=float,
    )

    if len(program_coefficients) != len(PROGRAM_COLUMNS):
        raise RuntimeError(
            "Unexpected program-coefficient dimension."
        )

    lineage_effect_map = {
        reference_lineage: 0.0,
    }

    lineage_effect_map.update(
        {
            lineage: float(coefficient)
            for lineage, coefficient in zip(
                encoded_lineages,
                lineage_coefficients,
            )
        }
    )

    train_lineage_counts = (
        train_df[LINEAGE_COLUMN]
        .value_counts()
    )

    train_mean_lineage_effect = sum(
        lineage_effect_map[lineage]
        * train_lineage_counts.loc[lineage]
        / len(train_df)
        for lineage in lineage_categories
    )

    fold_parameters = {
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "n_train_lineages": int(len(lineage_categories)),
        "program_intercept": float(
            linear_model.intercept_
        ),
        "reference_lineage": reference_lineage,
        "train_mean_lineage_effect": float(
            train_mean_lineage_effect
        ),
    }

    for program, coefficient in zip(
        PROGRAM_COLUMNS,
        program_coefficients,
    ):
        fold_parameters[
            f"beta_{program}"
        ] = float(coefficient)

        fold_parameters[
            f"train_mean_{program}"
        ] = float(train_df[program].mean())

    lineage_effect_rows = []

    for lineage in lineage_categories:
        n_train_lineage = int(
            train_lineage_counts.loc[lineage]
        )

        lineage_effect_rows.append(
            {
                LINEAGE_COLUMN: lineage,
                "lineage_effect": float(
                    lineage_effect_map[lineage]
                ),
                "n_train_lineage": n_train_lineage,
                "train_lineage_fraction": (
                    n_train_lineage / len(train_df)
                ),
                "is_reference_lineage": (
                    lineage == reference_lineage
                ),
            }
        )

    lineage_effects = pd.DataFrame(
        lineage_effect_rows
    )

    return (
        predictions,
        fold_parameters,
        lineage_effects,
    )

In [13]:
# =============================================================================
# Define repeat-level out-of-fold performance metrics
# =============================================================================

def summarize_repeat_performance(oof_predictions):
    observed = oof_predictions[TARGET_COLUMN]

    r2_lineage = r2_score(
        observed,
        oof_predictions["prediction_lineage"],
    )
    r2_program = r2_score(
        observed,
        oof_predictions["prediction_program"],
    )

    rmse_lineage = np.sqrt(
        mean_squared_error(
            observed,
            oof_predictions["prediction_lineage"],
        )
    )
    rmse_program = np.sqrt(
        mean_squared_error(
            observed,
            oof_predictions["prediction_program"],
        )
    )

    mae_lineage = mean_absolute_error(
        observed,
        oof_predictions["prediction_lineage"],
    )
    mae_program = mean_absolute_error(
        observed,
        oof_predictions["prediction_program"],
    )

    return {
        "r2_lineage": r2_lineage,
        "r2_program": r2_program,
        "delta_r2": r2_program - r2_lineage,
        "rmse_lineage": rmse_lineage,
        "rmse_program": rmse_program,
        "mae_lineage": mae_lineage,
        "mae_program": mae_program,
    }

In [14]:
# =============================================================================
# Smoke-test one primary CV fold
# =============================================================================

test_drug_id = sorted(gdsc_modeling[DRUG_ID_COLUMN].unique())[0]
test_repeat = 1
test_fold = 1

drug_df = gdsc_modeling.loc[
    gdsc_modeling[DRUG_ID_COLUMN].eq(test_drug_id)
].copy()

test_model_ids = set(
    cv_partitions.loc[
        cv_partitions[DRUG_ID_COLUMN].eq(test_drug_id)
        & cv_partitions["repeat"].eq(test_repeat)
        & cv_partitions["fold"].eq(test_fold),
        MODEL_ID_COLUMN,
    ]
)

train_df = drug_df.loc[
    ~drug_df[MODEL_ID_COLUMN].isin(test_model_ids)
].copy()

test_df = drug_df.loc[
    drug_df[MODEL_ID_COLUMN].isin(test_model_ids)
].copy()

smoke_predictions = fit_predict_primary_fold(
    train_df=train_df,
    test_df=test_df,
)

print("Drug ID:", test_drug_id)
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Predicted rows:", len(smoke_predictions))
print(
    "Finite predictions:",
    np.isfinite(
        smoke_predictions[
            ["prediction_lineage", "prediction_program"]
        ].to_numpy()
    ).all(),
)

Drug ID: 1003
Training rows: 444
Test rows: 112
Predicted rows: 112
Finite predictions: True


In [15]:
# =============================================================================
# Define drug-level repeated out-of-fold prediction
# =============================================================================

def generate_drug_oof_predictions(
    drug_id,
    return_state=False,
):
    drug_df = gdsc_modeling.loc[
        gdsc_modeling[DRUG_ID_COLUMN].eq(drug_id)
    ].copy()

    drug_partitions = cv_partitions.loc[
        cv_partitions[DRUG_ID_COLUMN].eq(drug_id)
    ]

    prediction_rows = []
    fold_parameter_rows = []
    lineage_effect_rows = []

    for repeat in range(1, N_REPEATS + 1):
        for fold in range(1, N_SPLITS + 1):
            test_ids = set(
                drug_partitions.loc[
                    drug_partitions["repeat"].eq(repeat)
                    & drug_partitions["fold"].eq(fold),
                    MODEL_ID_COLUMN,
                ]
            )

            train_df = drug_df.loc[
                ~drug_df[MODEL_ID_COLUMN].isin(test_ids)
            ]

            test_df = drug_df.loc[
                drug_df[MODEL_ID_COLUMN].isin(test_ids)
            ]

            if return_state:
                (
                    fold_predictions,
                    fold_parameters,
                    fold_lineage_effects,
                ) = fit_predict_primary_fold(
                    train_df=train_df,
                    test_df=test_df,
                    return_state=True,
                )

                fold_parameter_rows.append(
                    {
                        DRUG_ID_COLUMN: drug_id,
                        "repeat": repeat,
                        "fold": fold,
                        **fold_parameters,
                    }
                )

                fold_lineage_effects.insert(
                    0,
                    DRUG_ID_COLUMN,
                    drug_id,
                )
                fold_lineage_effects.insert(
                    1,
                    "repeat",
                    repeat,
                )
                fold_lineage_effects.insert(
                    2,
                    "fold",
                    fold,
                )

                lineage_effect_rows.append(
                    fold_lineage_effects
                )

            else:
                fold_predictions = fit_predict_primary_fold(
                    train_df=train_df,
                    test_df=test_df,
                )

            fold_predictions["repeat"] = repeat
            fold_predictions["fold"] = fold

            prediction_rows.append(
                fold_predictions
            )

    predictions = pd.concat(
        prediction_rows,
        ignore_index=True,
    )

    if not return_state:
        return predictions

    fold_parameters = pd.DataFrame(
        fold_parameter_rows
    )

    lineage_effects = pd.concat(
        lineage_effect_rows,
        ignore_index=True,
    )

    return (
        predictions,
        fold_parameters,
        lineage_effects,
    )

In [16]:
# =============================================================================
# Validate complete repeated OOF prediction for one drug
# =============================================================================

test_drug_oof = generate_drug_oof_predictions(test_drug_id)

expected_oof_rows = len(drug_df) * N_REPEATS

print("Drug ID:", test_drug_id)
print("OOF prediction rows:", len(test_drug_oof))
print("Expected rows:", expected_oof_rows)

print(
    "One OOF prediction per ModelID × repeat:",
    not test_drug_oof.duplicated(
        [MODEL_ID_COLUMN, "repeat"]
    ).any(),
)

print(
    "Models represented in every repeat:",
    test_drug_oof.groupby(MODEL_ID_COLUMN)["repeat"]
    .nunique()
    .eq(N_REPEATS)
    .all(),
)

print(
    "Finite predictions:",
    np.isfinite(
        test_drug_oof[
            ["prediction_lineage", "prediction_program"]
        ].to_numpy()
    ).all(),
)

Drug ID: 1003
OOF prediction rows: 2780
Expected rows: 2780
One OOF prediction per ModelID × repeat: True
Models represented in every repeat: True
Finite predictions: True


In [17]:
# =============================================================================
# Summarize repeat-level performance for one drug
# =============================================================================

test_drug_performance = pd.DataFrame(
    [
        {
            DRUG_ID_COLUMN: test_drug_id,
            "repeat": repeat,
            **summarize_repeat_performance(
                test_drug_oof.loc[
                    test_drug_oof["repeat"].eq(repeat)
                ]
            ),
        }
        for repeat in range(1, N_REPEATS + 1)
    ]
)

test_drug_performance

,DRUG_ID,repeat,r2_lineage,r2_program,delta_r2,rmse_lineage,rmse_program,mae_lineage,mae_program
0,1003,1,0.241099,0.265669,0.024570,1.650012,1.623082,1.314599,1.294795
1,1003,2,0.233020,0.257784,0.024765,1.658772,1.631773,1.318873,1.296163
2,1003,3,0.230074,0.255252,0.025177,1.661954,1.634554,1.324898,1.302995
3,1003,4,0.240276,0.261895,0.021619,1.650906,1.627248,1.314404,1.299486
4,1003,5,0.233559,0.258320,0.024761,1.658188,1.631183,1.320885,1.301932


In [18]:
# =============================================================================
# Define drug-level predictive-validity summary
# =============================================================================

def summarize_drug_performance(repeat_performance):
    median_r2_lineage = repeat_performance["r2_lineage"].median()
    median_r2_program = repeat_performance["r2_program"].median()
    median_delta_r2 = repeat_performance["delta_r2"].median()
    positive_delta_repeats = repeat_performance["delta_r2"].gt(0).sum()

    return {
        "median_r2_lineage": median_r2_lineage,
        "median_r2_program": median_r2_program,
        "median_delta_r2": median_delta_r2,
        "positive_delta_repeats": positive_delta_repeats,
        "shap_eligible": (
            median_r2_program >= MIN_MEDIAN_R2_PROGRAM
            and median_delta_r2 >= MIN_MEDIAN_DELTA_R2
            and positive_delta_repeats >= MIN_POSITIVE_DELTA_REPEATS
        ),
    }


test_drug_summary = summarize_drug_performance(
    test_drug_performance
)

test_drug_summary

{'median_r2_lineage': np.float64(0.2335593283909031),
 'median_r2_program': np.float64(0.25832044962543665),
 'median_delta_r2': np.float64(0.024761121234533556),
 'positive_delta_repeats': np.int64(5),
 'shap_eligible': np.True_}

In [19]:
# =============================================================================
# Finalize drug-level predictive-performance summary
# =============================================================================

def summarize_drug_performance(repeat_performance):
    median_r2_lineage = repeat_performance["r2_lineage"].median()
    median_r2_program = repeat_performance["r2_program"].median()
    median_delta_r2 = repeat_performance["delta_r2"].median()

    median_rmse_lineage = repeat_performance["rmse_lineage"].median()
    median_rmse_program = repeat_performance["rmse_program"].median()

    median_mae_lineage = repeat_performance["mae_lineage"].median()
    median_mae_program = repeat_performance["mae_program"].median()

    positive_delta_repeats = repeat_performance["delta_r2"].gt(0).sum()

    return {
        "median_r2_lineage": median_r2_lineage,
        "median_r2_program": median_r2_program,
        "median_delta_r2": median_delta_r2,
        "median_rmse_lineage": median_rmse_lineage,
        "median_rmse_program": median_rmse_program,
        "median_mae_lineage": median_mae_lineage,
        "median_mae_program": median_mae_program,
        "positive_delta_repeats": positive_delta_repeats,
        "shap_eligible": (
            median_r2_program >= MIN_MEDIAN_R2_PROGRAM
            and median_delta_r2 >= MIN_MEDIAN_DELTA_R2
            and positive_delta_repeats >= MIN_POSITIVE_DELTA_REPEATS
        ),
    }


test_drug_summary = summarize_drug_performance(
    test_drug_performance
)

test_drug_summary

{'median_r2_lineage': np.float64(0.2335593283909031),
 'median_r2_program': np.float64(0.25832044962543665),
 'median_delta_r2': np.float64(0.024761121234533556),
 'median_rmse_lineage': np.float64(1.6581882330229873),
 'median_rmse_program': np.float64(1.6311830876136524),
 'median_mae_lineage': np.float64(1.3188730987653836),
 'median_mae_program': np.float64(1.2994863905354443),
 'positive_delta_repeats': np.int64(5),
 'shap_eligible': np.True_}

In [20]:
# =============================================================================
# Execute primary repeated OOF evaluation across all eligible GDSC drugs
# =============================================================================

repeat_performance_rows = []
drug_summary_rows = []

primary_oof_rows = []
primary_program_fold_parameter_rows = []
primary_program_fold_lineage_effect_rows = []

drug_ids = sorted(
    gdsc_modeling[DRUG_ID_COLUMN].unique()
)

for drug_index, drug_id in enumerate(
    drug_ids,
    start=1,
):
    (
        drug_oof,
        drug_fold_parameters,
        drug_fold_lineage_effects,
    ) = generate_drug_oof_predictions(
        drug_id,
        return_state=True,
    )

    primary_oof_rows.append(
        drug_oof
    )

    primary_program_fold_parameter_rows.append(
        drug_fold_parameters
    )

    primary_program_fold_lineage_effect_rows.append(
        drug_fold_lineage_effects
    )

    drug_repeat_performance = pd.DataFrame(
        [
            {
                DRUG_ID_COLUMN: drug_id,
                "repeat": repeat,
                **summarize_repeat_performance(
                    drug_oof.loc[
                        drug_oof["repeat"].eq(repeat)
                    ]
                ),
            }
            for repeat in range(1, N_REPEATS + 1)
        ]
    )

    repeat_performance_rows.append(
        drug_repeat_performance
    )

    drug_summary_rows.append(
        {
            DRUG_ID_COLUMN: drug_id,
            **summarize_drug_performance(
                drug_repeat_performance
            ),
        }
    )

    if (
        drug_index % 25 == 0
        or drug_index == len(drug_ids)
    ):
        print(
            f"Completed {drug_index}/{len(drug_ids)} drugs"
        )

primary_oof_predictions = pd.concat(
    primary_oof_rows,
    ignore_index=True,
)

primary_program_fold_parameters = pd.concat(
    primary_program_fold_parameter_rows,
    ignore_index=True,
)

primary_program_fold_lineage_effects = pd.concat(
    primary_program_fold_lineage_effect_rows,
    ignore_index=True,
)

primary_repeat_performance = pd.concat(
    repeat_performance_rows,
    ignore_index=True,
)

primary_drug_summary = pd.DataFrame(
    drug_summary_rows
)

print(
    "\nOOF prediction shape:",
    primary_oof_predictions.shape,
)
print(
    "Fold-parameter shape:",
    primary_program_fold_parameters.shape,
)
print(
    "Fold-lineage-effect shape:",
    primary_program_fold_lineage_effects.shape,
)
print(
    "Repeat-level performance shape:",
    primary_repeat_performance.shape,
)
print(
    "Drug-level summary shape:",
    primary_drug_summary.shape,
)

Completed 25/281 drugs
Completed 50/281 drugs
Completed 75/281 drugs
Completed 100/281 drugs
Completed 125/281 drugs
Completed 150/281 drugs
Completed 175/281 drugs
Completed 200/281 drugs
Completed 225/281 drugs
Completed 250/281 drugs
Completed 275/281 drugs
Completed 281/281 drugs

OOF prediction shape: (680880, 11)
Fold-parameter shape: (7025, 15)
Fold-lineage-effect shape: (74025, 8)
Repeat-level performance shape: (1405, 9)
Drug-level summary shape: (281, 10)


In [21]:
# =============================================================================
# Validate completeness of primary predictive evaluation and fitted-state handoff
# =============================================================================

metric_columns = [
    "r2_lineage",
    "r2_program",
    "delta_r2",
    "rmse_lineage",
    "rmse_program",
    "mae_lineage",
    "mae_program",
]

expected_oof_rows = (
    len(gdsc_modeling) * N_REPEATS
)

expected_fold_rows = (
    len(drug_ids)
    * N_REPEATS
    * N_SPLITS
)

expected_lineage_effect_rows = int(
    gdsc_modeling
    .groupby(DRUG_ID_COLUMN)[LINEAGE_COLUMN]
    .nunique()
    .sum()
    * N_REPEATS
    * N_SPLITS
)

print(
    "Exactly five repeats per drug:",
    primary_repeat_performance
    .groupby(DRUG_ID_COLUMN)["repeat"]
    .nunique()
    .eq(N_REPEATS)
    .all(),
)

print(
    "All repeat-level metrics finite:",
    np.isfinite(
        primary_repeat_performance[
            metric_columns
        ].to_numpy()
    ).all(),
)

print(
    "All 281 drugs represented in summary:",
    primary_drug_summary[
        DRUG_ID_COLUMN
    ].nunique() == len(drug_ids),
)

print(
    "OOF rows:",
    len(primary_oof_predictions),
    "/ expected:",
    expected_oof_rows,
)

print(
    "One OOF row per drug × ModelID × repeat:",
    not primary_oof_predictions.duplicated(
        [
            DRUG_ID_COLUMN,
            MODEL_ID_COLUMN,
            "repeat",
        ]
    ).any(),
)

print(
    "Fold-state rows:",
    len(primary_program_fold_parameters),
    "/ expected:",
    expected_fold_rows,
)

print(
    "One state per drug × repeat × fold:",
    not primary_program_fold_parameters.duplicated(
        [
            DRUG_ID_COLUMN,
            "repeat",
            "fold",
        ]
    ).any(),
)

print(
    "Fold-lineage-effect rows:",
    len(primary_program_fold_lineage_effects),
    "/ expected:",
    expected_lineage_effect_rows,
)

print(
    "One lineage effect per drug × repeat × fold × lineage:",
    not primary_program_fold_lineage_effects.duplicated(
        [
            DRUG_ID_COLUMN,
            "repeat",
            "fold",
            LINEAGE_COLUMN,
        ]
    ).any(),
)

reference_counts = (
    primary_program_fold_lineage_effects
    .groupby(
        [
            DRUG_ID_COLUMN,
            "repeat",
            "fold",
        ],
        observed=True,
    )["is_reference_lineage"]
    .sum()
)

print(
    "Exactly one reference lineage per fitted model:",
    reference_counts.eq(1).all(),
)

lineage_fraction_sums = (
    primary_program_fold_lineage_effects
    .groupby(
        [
            DRUG_ID_COLUMN,
            "repeat",
            "fold",
        ],
        observed=True,
    )["train_lineage_fraction"]
    .sum()
)

print(
    "Training lineage fractions sum to one:",
    np.allclose(
        lineage_fraction_sums.to_numpy(),
        1.0,
    ),
)

Exactly five repeats per drug: True
All repeat-level metrics finite: True
All 281 drugs represented in summary: True
OOF rows: 680880 / expected: 680880
One OOF row per drug × ModelID × repeat: True
Fold-state rows: 7025 / expected: 7025
One state per drug × repeat × fold: True
Fold-lineage-effect rows: 74025 / expected: 74025
One lineage effect per drug × repeat × fold × lineage: True
Exactly one reference lineage per fitted model: True
Training lineage fractions sum to one: True


In [22]:
# =============================================================================
# Summarize frozen predictive-validity gate outcomes
# =============================================================================

primary_drug_summary = primary_drug_summary.assign(
    passes_r2_program=lambda df: (
        df["median_r2_program"] >= MIN_MEDIAN_R2_PROGRAM
    ),
    passes_delta_r2=lambda df: (
        df["median_delta_r2"] >= MIN_MEDIAN_DELTA_R2
    ),
    passes_delta_stability=lambda df: (
        df["positive_delta_repeats"] >= MIN_POSITIVE_DELTA_REPEATS
    ),
)

gate_summary = pd.Series(
    {
        "total_drugs": len(primary_drug_summary),
        "passes_r2_program": primary_drug_summary["passes_r2_program"].sum(),
        "passes_delta_r2": primary_drug_summary["passes_delta_r2"].sum(),
        "passes_delta_stability": primary_drug_summary[
            "passes_delta_stability"
        ].sum(),
        "passes_all_criteria": primary_drug_summary["shap_eligible"].sum(),
    }
)

gate_summary

total_drugs               281
passes_r2_program         275
passes_delta_r2           126
passes_delta_stability    226
passes_all_criteria       125
dtype: int64

In [23]:
# =============================================================================
# Characterize predictive-validity gate outcomes
# =============================================================================

def classify_gate_outcome(row):
    failed = []

    if not row["passes_r2_program"]:
        failed.append("overall_predictive_validity")

    if not row["passes_delta_r2"]:
        failed.append("incremental_program_contribution")

    if not row["passes_delta_stability"]:
        failed.append("incremental_stability")

    if not failed:
        return "eligible_for_primary_shap"

    return " + ".join(failed)


primary_drug_summary["gate_outcome"] = primary_drug_summary.apply(
    classify_gate_outcome,
    axis=1,
)

gate_outcome_summary = (
    primary_drug_summary["gate_outcome"]
    .value_counts()
    .rename_axis("gate_outcome")
    .reset_index(name="n_drugs")
)

gate_outcome_summary["percent_drugs"] = (
    100
    * gate_outcome_summary["n_drugs"]
    / len(primary_drug_summary)
)

gate_outcome_summary

,gate_outcome,n_drugs,percent_drugs
0,eligible_for_primary_shap,125,44.483986
1,incremental_program_contribution,99,35.231317
2,incremental_program_contribution + incremental...,51,18.149466
3,overall_predictive_validity + incremental_prog...,4,1.423488
4,overall_predictive_validity,1,0.355872
5,overall_predictive_validity + incremental_prog...,1,0.355872


In [24]:
# =============================================================================
# Summarize primary predictive-performance distributions
# =============================================================================

performance_distribution = (
    primary_drug_summary[
        [
            "median_r2_lineage",
            "median_r2_program",
            "median_delta_r2",
            "median_rmse_lineage",
            "median_rmse_program",
            "median_mae_lineage",
            "median_mae_program",
        ]
    ]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .T
)

performance_distribution

,count,mean,std,min,10%,25%,50%,75%,90%,max
median_r2_lineage,281.0,0.228935,0.094216,0.013983,0.109882,0.160134,0.231323,0.301551,0.346793,0.439188
median_r2_program,281.0,0.248845,0.098381,0.000721,0.120557,0.177164,0.243832,0.321123,0.374686,0.473930
median_delta_r2,281.0,0.020372,0.021965,-0.015253,-0.003464,0.005843,0.015957,0.031943,0.045963,0.145260
median_rmse_lineage,281.0,1.267768,0.378855,0.689072,0.820487,0.953385,1.249401,1.477827,1.740131,2.787871
median_rmse_program,281.0,1.251045,0.374984,0.687584,0.813112,0.947888,1.225713,1.462314,1.703674,2.762817
median_mae_lineage,281.0,0.988686,0.305751,0.493126,0.645611,0.736515,0.969615,1.153646,1.347159,2.271884
median_mae_program,281.0,0.975480,0.301477,0.491685,0.634383,0.728679,0.957857,1.141379,1.344292,2.237444


## Primary predictive-performance summary

Across the 281 prospectively eligible GDSC drugs, lineage alone captured a substantial component of out-of-fold predictive structure.

The median lineage-only performance was `R² = 0.231`, compared with `R² = 0.244` for the lineage-plus-program model. The median incremental contribution of the three frozen consensus programs was therefore modest (`median ΔR² = 0.016`) and heterogeneous across compounds.

Incremental performance was not uniformly positive or substantial: the 10th percentile of drug-level `median ΔR²` was slightly below zero, whereas the 75th and 90th percentiles were approximately `0.032` and `0.046`, respectively.

Under the prospectively frozen predictive-validity gate, 125 of 281 drugs (44.5%) satisfied all criteria required for subsequent primary program-level SHAP interpretation. The most common reason for gate failure was insufficient incremental predictive contribution beyond lineage rather than inadequate overall predictive performance.

These results support a distinction between general predictive structure attributable to lineage and additional predictive information associated with the frozen consensus programs. They do not establish biological mechanism, causality, therapeutic relevance, external reproducibility, or clinical predictiveness.

No drug ranking or post hoc threshold modification is introduced from these results.


In [25]:
# =============================================================================
# Define secondary unseen-lineage stress-test models
# =============================================================================

def fit_predict_lolo_fold(train_df, test_df):
    intercept_model = LinearRegression()
    program_model = LinearRegression()

    intercept_model.fit(
        np.ones((len(train_df), 1)),
        train_df[TARGET_COLUMN],
    )
    program_model.fit(
        train_df[PROGRAM_COLUMNS],
        train_df[TARGET_COLUMN],
    )

    predictions = test_df[
        [DRUG_ID_COLUMN, MODEL_ID_COLUMN, LINEAGE_COLUMN, TARGET_COLUMN]
    ].copy()

    predictions["prediction_intercept"] = intercept_model.predict(
        np.ones((len(test_df), 1))
    )
    predictions["prediction_program"] = program_model.predict(
        test_df[PROGRAM_COLUMNS]
    )

    return predictions

In [26]:
# =============================================================================
# Define unseen-lineage stress-test performance metrics
# =============================================================================

def summarize_lolo_performance(predictions):
    observed = predictions[TARGET_COLUMN]

    r2_intercept = r2_score(
        observed,
        predictions["prediction_intercept"],
    )
    r2_program = r2_score(
        observed,
        predictions["prediction_program"],
    )

    rmse_intercept = np.sqrt(
        mean_squared_error(
            observed,
            predictions["prediction_intercept"],
        )
    )
    rmse_program = np.sqrt(
        mean_squared_error(
            observed,
            predictions["prediction_program"],
        )
    )

    mae_intercept = mean_absolute_error(
        observed,
        predictions["prediction_intercept"],
    )
    mae_program = mean_absolute_error(
        observed,
        predictions["prediction_program"],
    )

    return {
        "r2_intercept": r2_intercept,
        "r2_program": r2_program,
        "delta_r2": r2_program - r2_intercept,
        "rmse_intercept": rmse_intercept,
        "rmse_program": rmse_program,
        "mae_intercept": mae_intercept,
        "mae_program": mae_program,
    }

In [27]:
# =============================================================================
# Smoke-test one unseen-lineage stress-test fold
# =============================================================================

test_lolo_drug_id = sorted(gdsc_modeling[DRUG_ID_COLUMN].unique())[0]

test_lolo_drug_df = gdsc_modeling.loc[
    gdsc_modeling[DRUG_ID_COLUMN].eq(test_lolo_drug_id)
].copy()

test_lolo_lineage = sorted(
    test_lolo_drug_df[LINEAGE_COLUMN].unique()
)[0]

lolo_train_df = test_lolo_drug_df.loc[
    ~test_lolo_drug_df[LINEAGE_COLUMN].eq(test_lolo_lineage)
].copy()

lolo_test_df = test_lolo_drug_df.loc[
    test_lolo_drug_df[LINEAGE_COLUMN].eq(test_lolo_lineage)
].copy()

lolo_smoke_predictions = fit_predict_lolo_fold(
    train_df=lolo_train_df,
    test_df=lolo_test_df,
)

print("Drug ID:", test_lolo_drug_id)
print("Held-out lineage:", test_lolo_lineage)
print("Training rows:", len(lolo_train_df))
print("Test rows:", len(lolo_test_df))
print("Predicted rows:", len(lolo_smoke_predictions))
print(
    "Finite predictions:",
    np.isfinite(
        lolo_smoke_predictions[
            ["prediction_intercept", "prediction_program"]
        ].to_numpy()
    ).all(),
)

Drug ID: 1003
Held-out lineage: Bowel
Training rows: 513
Test rows: 43
Predicted rows: 43
Finite predictions: True


## Secondary unseen-lineage stress test

The prospectively defined leave-one-supported-lineage-out analysis evaluates descriptive transportability of the frozen consensus programs to a lineage that is completely absent during model fitting.

For each eligible GDSC drug and each supported lineage:

* the complete lineage is held out;
* the remaining supported lineages form the training set;
* the reference model is intercept-only;
* the program model contains `CONSENSUS_TX_01`, `CONSENSUS_TX_02`, and `CONSENSUS_TX_03`;
* no lineage indicator is fitted because the test lineage is absent from training.

Performance is retained separately for every `drug × held-out lineage` using:

* `R²`;
* RMSE;
* MAE; and
* `ΔR² = R²_program − R²_intercept`.

Drug-level descriptive summaries will use the median across held-out lineages and the fraction of held-out lineages with positive `ΔR²`.

No LOLO threshold defines eligibility for notebook 602.

LOLO results must not:

* replace the primary repeated lineage-stratified evaluation;
* rescue a drug that fails the frozen primary predictive-validity gate;
* remove a drug that passes the primary gate;
* define a new favorable-drug subset; or
* alter the frozen primary modeling specification.

Negative or heterogeneous unseen-lineage transportability is retained as a valid secondary result.


In [28]:
# =============================================================================
# Execute secondary unseen-lineage stress test across all eligible drugs
# =============================================================================

lolo_performance_rows = []

for drug_index, drug_id in enumerate(drug_ids, start=1):
    drug_df = gdsc_modeling.loc[
        gdsc_modeling[DRUG_ID_COLUMN].eq(drug_id)
    ].copy()

    for held_out_lineage in sorted(
        drug_df[LINEAGE_COLUMN].unique()
    ):
        train_df = drug_df.loc[
            ~drug_df[LINEAGE_COLUMN].eq(held_out_lineage)
        ].copy()

        test_df = drug_df.loc[
            drug_df[LINEAGE_COLUMN].eq(held_out_lineage)
        ].copy()

        predictions = fit_predict_lolo_fold(
            train_df=train_df,
            test_df=test_df,
        )

        lolo_performance_rows.append(
            {
                DRUG_ID_COLUMN: drug_id,
                "held_out_lineage": held_out_lineage,
                "n_train": len(train_df),
                "n_test": len(test_df),
                **summarize_lolo_performance(predictions),
            }
        )

    if drug_index % 25 == 0 or drug_index == len(drug_ids):
        print(
            f"Completed {drug_index}/{len(drug_ids)} drugs"
        )

lolo_performance = pd.DataFrame(
    lolo_performance_rows
)

print("\nLOLO performance shape:", lolo_performance.shape)
print(
    "Unique drugs:",
    lolo_performance[DRUG_ID_COLUMN].nunique(),
)

Completed 25/281 drugs
Completed 50/281 drugs
Completed 75/281 drugs
Completed 100/281 drugs
Completed 125/281 drugs
Completed 150/281 drugs
Completed 175/281 drugs
Completed 200/281 drugs
Completed 225/281 drugs
Completed 250/281 drugs
Completed 275/281 drugs
Completed 281/281 drugs

LOLO performance shape: (2961, 11)
Unique drugs: 281


In [29]:
# =============================================================================
# Summarize unseen-lineage stress-test performance by drug
# =============================================================================

lolo_drug_summary = (
    lolo_performance
    .groupby(DRUG_ID_COLUMN, as_index=False)
    .agg(
        n_held_out_lineages=("held_out_lineage", "nunique"),
        median_r2_intercept=("r2_intercept", "median"),
        median_r2_program=("r2_program", "median"),
        median_delta_r2=("delta_r2", "median"),
        median_rmse_intercept=("rmse_intercept", "median"),
        median_rmse_program=("rmse_program", "median"),
        median_mae_intercept=("mae_intercept", "median"),
        median_mae_program=("mae_program", "median"),
        positive_delta_lineages=(
            "delta_r2",
            lambda values: int((values > 0).sum()),
        ),
        fraction_positive_delta_lineages=(
            "delta_r2",
            lambda values: float((values > 0).mean()),
        ),
    )
)

print("Drug-level LOLO summary shape:", lolo_drug_summary.shape)

lolo_drug_summary[
    [
        "n_held_out_lineages",
        "median_r2_intercept",
        "median_r2_program",
        "median_delta_r2",
        "fraction_positive_delta_lineages",
    ]
].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
).T

Drug-level LOLO summary shape: (281, 11)


,count,mean,std,min,10%,25%,50%,75%,90%,max
n_held_out_lineages,281.0,10.537367,0.750851,4.000000,10.000000,10.000000,11.000000,11.000000,11.000000,11.000000
median_r2_intercept,281.0,-0.203046,0.116275,-0.693904,-0.351836,-0.263937,-0.176108,-0.123954,-0.085985,-0.020306
median_r2_program,281.0,-0.116703,0.131738,-1.025177,-0.229414,-0.147452,-0.087877,-0.043388,-0.021240,0.052314
median_delta_r2,281.0,0.162379,0.122679,-0.344123,0.030628,0.093811,0.157135,0.225132,0.309280,0.562629
fraction_positive_delta_lineages,281.0,0.726040,0.151225,0.000000,0.545455,0.636364,0.727273,0.818182,0.900000,1.000000


## Secondary unseen-lineage stress-test results

Unseen-lineage transportability was substantially weaker than the primary within-supported-lineage predictive setting.

Across the 281 eligible GDSC drugs, the median drug-level `R²` of the program model was negative (`median = -0.088`), indicating limited absolute predictive transportability when an entire lineage was absent during fitting. The corresponding intercept-only reference was poorer (`median R² = -0.176`).

Relative to that training-derived intercept-only reference, the frozen consensus programs frequently improved prediction. The median drug-level `ΔR²` was `0.157`, and the median fraction of held-out lineages with positive `ΔR²` was `0.727`.

These two observations are not contradictory. Negative absolute `R²` indicates that unseen-lineage prediction remains difficult, whereas positive `ΔR²` indicates that the frozen programs can retain some predictive information relative to a model containing no lineage or transcriptomic information.

The unseen-lineage analysis therefore provides evidence of heterogeneous relative transportability rather than strong absolute generalization to completely unseen cancer lineages.

These results remain secondary descriptive evidence. They do not modify the primary predictive-validity gate, do not determine eligibility for notebook 602, and do not rescue or exclude individual drugs.

No causal, mechanistic, external-replication, therapeutic, or clinical interpretation is assigned to this stress test.


In [30]:
# =============================================================================
# Build integrated notebook-601 drug-level summary
# =============================================================================

lolo_summary_for_merge = lolo_drug_summary.rename(
    columns={
        "n_held_out_lineages": "lolo_n_held_out_lineages",
        "median_r2_intercept": "lolo_median_r2_intercept",
        "median_r2_program": "lolo_median_r2_program",
        "median_delta_r2": "lolo_median_delta_r2",
        "median_rmse_intercept": "lolo_median_rmse_intercept",
        "median_rmse_program": "lolo_median_rmse_program",
        "median_mae_intercept": "lolo_median_mae_intercept",
        "median_mae_program": "lolo_median_mae_program",
        "positive_delta_lineages": "lolo_positive_delta_lineages",
        "fraction_positive_delta_lineages": "lolo_fraction_positive_delta_lineages",
    }
)

drug_level_results = primary_drug_summary.merge(
    lolo_summary_for_merge,
    on=DRUG_ID_COLUMN,
    how="left",
    validate="one_to_one",
)

print("Integrated drug-level summary shape:", drug_level_results.shape)
print(
    "Drugs with complete LOLO summary:",
    drug_level_results["lolo_n_held_out_lineages"].notna().sum(),
)
print(
    "Primary SHAP-eligible drugs:",
    drug_level_results["shap_eligible"].sum(),
)

Integrated drug-level summary shape: (281, 24)
Drugs with complete LOLO summary: 281
Primary SHAP-eligible drugs: 125


In [31]:
# =============================================================================
# Persist stable notebook-601 tabular outputs
# =============================================================================

output_dir = Paths.pharmacogenomic_contexts
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_paths = {
    "cv_partitions": (
        output_dir
        / "601_primary_cv_partitions.parquet"
    ),
    "primary_oof_predictions": (
        output_dir
        / "601_primary_oof_predictions.parquet"
    ),
    "primary_program_fold_parameters": (
        output_dir
        / "601_primary_program_fold_parameters.parquet"
    ),
    "primary_program_fold_lineage_effects": (
        output_dir
        / "601_primary_program_fold_lineage_effects.parquet"
    ),
    "primary_repeat_performance": (
        output_dir
        / "601_primary_repeat_performance.csv"
    ),
    "lolo_performance": (
        output_dir
        / "601_lolo_performance.csv"
    ),
    "drug_level_results": (
        output_dir
        / "601_drug_level_results.csv"
    ),
}

cv_partitions.to_parquet(
    output_paths["cv_partitions"],
    index=False,
)

primary_oof_predictions.to_parquet(
    output_paths["primary_oof_predictions"],
    index=False,
)

primary_program_fold_parameters.to_parquet(
    output_paths[
        "primary_program_fold_parameters"
    ],
    index=False,
)

primary_program_fold_lineage_effects.to_parquet(
    output_paths[
        "primary_program_fold_lineage_effects"
    ],
    index=False,
)

primary_repeat_performance.to_csv(
    output_paths["primary_repeat_performance"],
    index=False,
)

lolo_performance.to_csv(
    output_paths["lolo_performance"],
    index=False,
)

drug_level_results.to_csv(
    output_paths["drug_level_results"],
    index=False,
)

for name, path in output_paths.items():
    print(
        f"{name}: "
        f"{project_relative_path(path)}"
    )

cv_partitions: data/processed/pharmacogenomic_contexts/601_primary_cv_partitions.parquet
primary_oof_predictions: data/processed/pharmacogenomic_contexts/601_primary_oof_predictions.parquet
primary_program_fold_parameters: data/processed/pharmacogenomic_contexts/601_primary_program_fold_parameters.parquet
primary_program_fold_lineage_effects: data/processed/pharmacogenomic_contexts/601_primary_program_fold_lineage_effects.parquet
primary_repeat_performance: data/processed/pharmacogenomic_contexts/601_primary_repeat_performance.csv
lolo_performance: data/processed/pharmacogenomic_contexts/601_lolo_performance.csv
drug_level_results: data/processed/pharmacogenomic_contexts/601_drug_level_results.csv


In [32]:
# =============================================================================
# Persist notebook-601 analysis metadata
# =============================================================================

import json

analysis_metadata = {
    "schema_version": 1,
    "notebook": "601_explainable_predictive_modeling",
    "analysis_role": "internal_predictive_evaluation",
    "resource": "GDSC",
    "input_artifact_ids": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.drug_eligibility",
        "phase6.600.program_score_universe",
    ],
    "target": {
        "column": TARGET_COLUMN,
        "metric": "LN_IC50",
        "direction": "higher_is_more_resistance_like",
    },
    "program_features": PROGRAM_COLUMNS,
    "primary_evaluation": {
        "estimand": (
            "previously unseen cell-line model from an already "
            "represented lineage for a known GDSC drug"
        ),
        "baseline_model": "LN_IC50 ~ C(OncotreeLineage)",
        "program_model": (
            "LN_IC50 ~ C(OncotreeLineage) + CONSENSUS_TX_01 "
            "+ CONSENSUS_TX_02 + CONSENSUS_TX_03"
        ),
        "model_family": "ordinary_linear_regression",
        "lineage_encoding": {
            "method": "one_hot",
            "drop": "first",
            "handle_unknown": "error",
        },
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "random_seed": RANDOM_SEED,
        "primary_metric": "median_delta_r2_across_repeats",
        "shap_gate": {
            "min_median_r2_program": MIN_MEDIAN_R2_PROGRAM,
            "min_median_delta_r2": MIN_MEDIAN_DELTA_R2,
            "min_positive_delta_repeats": MIN_POSITIVE_DELTA_REPEATS,
        },
    },
    "secondary_lolo": {
        "role": "descriptive_unseen_lineage_stress_test",
        "baseline_model": "LN_IC50 ~ 1",
        "program_model": (
            "LN_IC50 ~ CONSENSUS_TX_01 "
            "+ CONSENSUS_TX_02 + CONSENSUS_TX_03"
        ),
        "selection_role": "none",
    },
    "fitted_state_handoff": {
        "scope": "all_281_primary_gdsc_drugs",
        "purpose": (
            "preserve fitted primary-model state "
            "for downstream attribution without refitting"
        ),
        "downstream_refitting_required": False,
        "contains_oof_predictions": True,
        "contains_program_coefficients": True,
        "contains_training_program_means": True,
        "contains_lineage_effects": True,
        "contains_training_lineage_frequencies": True,
    },
    "execution_summary": {
        "eligible_gdsc_drugs": int(
            len(drug_level_results)
        ),
        "primary_oof_rows": int(
            len(primary_oof_predictions)
        ),
        "primary_program_fold_parameter_rows": int(
            len(primary_program_fold_parameters)
        ),
        "primary_program_fold_lineage_effect_rows": int(
            len(primary_program_fold_lineage_effects)
        ),
        "primary_repeat_rows": int(
            len(primary_repeat_performance)
        ),
        "primary_shap_eligible_drugs": int(
            drug_level_results["shap_eligible"].sum()
        ),
        "lolo_drug_lineage_evaluations": int(
            len(lolo_performance)
        ),
    },
    "evidence_isolation": {
        "ctrp_inspected_for_predictive_outcomes": False,
        "prism_inspected_for_predictive_outcomes": False,
        "phase5_used_for_selection": False,
        "notebook600_associations_used_for_selection": False,
    },
    "outputs": {
        name: path.relative_to(
            Paths.root
        ).as_posix()
        for name, path in output_paths.items()
    },
}

metadata_path = (
    output_dir
    / "601_analysis_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        analysis_metadata,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Analysis metadata:",
    project_relative_path(metadata_path),
)

print(
    "Recorded SHAP-eligible drugs:",
    analysis_metadata[
        "execution_summary"
    ][
        "primary_shap_eligible_drugs"
    ],
)

print(
    "Recorded primary OOF rows:",
    analysis_metadata[
        "execution_summary"
    ][
        "primary_oof_rows"
    ],
)

print(
    "Recorded fold-parameter rows:",
    analysis_metadata[
        "execution_summary"
    ][
        "primary_program_fold_parameter_rows"
    ],
)

print(
    "Recorded fold-lineage-effect rows:",
    analysis_metadata[
        "execution_summary"
    ][
        "primary_program_fold_lineage_effect_rows"
    ],
)

Analysis metadata: data/processed/pharmacogenomic_contexts/601_analysis_metadata.json
Recorded SHAP-eligible drugs: 125
Recorded primary OOF rows: 680880
Recorded fold-parameter rows: 7025
Recorded fold-lineage-effect rows: 74025


In [33]:
# =============================================================================
# Build notebook-601 artifact identities
# =============================================================================

from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)


artifact_outputs = {
    "phase6.601.primary_cv_partitions": {
        "path": output_paths["cv_partitions"],
        "shape": list(cv_partitions.shape),
        "artifact_role": "handoff",
    },
    "phase6.601.primary_oof_predictions": {
        "path": output_paths[
            "primary_oof_predictions"
        ],
        "shape": list(
            primary_oof_predictions.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.601.primary_program_fold_parameters": {
        "path": output_paths[
            "primary_program_fold_parameters"
        ],
        "shape": list(
            primary_program_fold_parameters.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.601.primary_program_fold_lineage_effects": {
        "path": output_paths[
            "primary_program_fold_lineage_effects"
        ],
        "shape": list(
            primary_program_fold_lineage_effects.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.601.primary_repeat_performance": {
        "path": output_paths[
            "primary_repeat_performance"
        ],
        "shape": list(
            primary_repeat_performance.shape
        ),
        "artifact_role": "data",
    },
    "phase6.601.lolo_performance": {
        "path": output_paths["lolo_performance"],
        "shape": list(
            lolo_performance.shape
        ),
        "artifact_role": "data",
    },
    "phase6.601.drug_level_results": {
        "path": output_paths[
            "drug_level_results"
        ],
        "shape": list(
            drug_level_results.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.601.analysis_metadata": {
        "path": metadata_path,
        "shape": None,
        "artifact_role": "metadata",
    },
}

artifact_identity_rows = []

for artifact_id, artifact in artifact_outputs.items():
    path = artifact["path"]

    artifact_identity_rows.append(
        {
            "artifact_id": artifact_id,
            "path": (
                path.relative_to(Paths.root)
                .as_posix()
            ),
            "artifact_role": artifact[
                "artifact_role"
            ],
            "shape": artifact["shape"],
            "size_bytes": path.stat().st_size,
            "sha256": calculate_sha256(path),
        }
    )

artifact_identities = pd.DataFrame(
    artifact_identity_rows
)

identity_by_id = (
    artifact_identities
    .set_index("artifact_id")
    .to_dict(orient="index")
)

preexisting_stable_ids = [
    "phase6.601.primary_cv_partitions",
    "phase6.601.primary_repeat_performance",
    "phase6.601.lolo_performance",
    "phase6.601.drug_level_results",
]

drift_findings = []

for artifact_id in preexisting_stable_ids:
    registered = artifact_registry[
        "artifacts"
    ][artifact_id]

    current = identity_by_id[artifact_id]

    if (
        current["size_bytes"]
        != registered["size_bytes"]
    ):
        drift_findings.append(
            {
                "artifact_id": artifact_id,
                "issue": "size_mismatch",
            }
        )

    if (
        current["sha256"]
        != registered["sha256"]
    ):
        drift_findings.append(
            {
                "artifact_id": artifact_id,
                "issue": "sha256_mismatch",
            }
        )

print(
    "Pre-existing analytical artifact "
    "drift findings:",
    drift_findings,
)

artifact_identities

Pre-existing analytical artifact drift findings: []


,artifact_id,path,artifact_role,shape,size_bytes,sha256
0,phase6.601.primary_cv_partitions,data/processed/pharmacogenomic_contexts/601_pr...,handoff,"[680880, 4]",838062,ccaf0aecae5b5a6b3e43b871ec9028a852038ab1746777...
1,phase6.601.primary_oof_predictions,data/processed/pharmacogenomic_contexts/601_pr...,handoff,"[680880, 11]",12722683,6e6e5a6138dfcc55d61ffe0791843c110425ef98f718ab...
2,phase6.601.primary_program_fold_parameters,data/processed/pharmacogenomic_contexts/601_pr...,handoff,"[7025, 15]",437090,d273bc2fc819e3521fe924c28d5106c3d85a186d088ea5...
3,phase6.601.primary_program_fold_lineage_effects,data/processed/pharmacogenomic_contexts/601_pr...,handoff,"[74025, 8]",749643,77850676d62e710693accf7262764bf877e36c157e49d8...
4,phase6.601.primary_repeat_performance,data/processed/pharmacogenomic_contexts/601_pr...,data,"[1405, 9]",201442,37d0bb623fbdf8100e2ecdd56bf60d2d1de70e76ee80a9...
5,phase6.601.lolo_performance,data/processed/pharmacogenomic_contexts/601_lo...,data,"[2961, 11]",471594,f8fd0f1d99d4b5b88c52845ede19a5c39e6b2eb8c084e2...
6,phase6.601.drug_level_results,data/processed/pharmacogenomic_contexts/601_dr...,handoff,"[281, 24]",100059,1ee129347eef574519fe61e0bf19381da4d6c731f3934b...
7,phase6.601.analysis_metadata,data/processed/pharmacogenomic_contexts/601_an...,metadata,None,3330,9377729df1423eee7f916273c4621935fb2d5fd6bc0055...


In [34]:
# =============================================================================
# Define notebook-601 artifact-registry entries
# =============================================================================

producer = {
    "type": "notebook",
    "path": (
        "notebooks/phase6_pharmacogenomic_contexts/"
        "601_explainable_predictive_modeling.ipynb"
    ),
}

artifact_inputs = {
    "phase6.601.primary_cv_partitions": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.drug_eligibility",
    ],
    "phase6.601.primary_oof_predictions": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.program_score_universe",
        "phase6.601.primary_cv_partitions",
    ],
    "phase6.601.primary_program_fold_parameters": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.program_score_universe",
        "phase6.601.primary_cv_partitions",
    ],
    "phase6.601.primary_program_fold_lineage_effects": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.program_score_universe",
        "phase6.601.primary_cv_partitions",
    ],
    "phase6.601.primary_repeat_performance": [
        "phase6.601.primary_oof_predictions",
    ],
    "phase6.601.lolo_performance": [
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.program_score_universe",
    ],
    "phase6.601.drug_level_results": [
        "phase6.601.primary_repeat_performance",
        "phase6.601.lolo_performance",
    ],
    "phase6.601.analysis_metadata": [
        "phase6.601.primary_cv_partitions",
        "phase6.601.primary_oof_predictions",
        "phase6.601.primary_program_fold_parameters",
        "phase6.601.primary_program_fold_lineage_effects",
        "phase6.601.primary_repeat_performance",
        "phase6.601.lolo_performance",
        "phase6.601.drug_level_results",
    ],
}

identity_by_id = (
    artifact_identities
    .set_index("artifact_id")
    .to_dict(orient="index")
)

registry_entries = {}

for artifact_id, identity in identity_by_id.items():
    shape = identity["shape"]

    registry_entries[artifact_id] = {
        "path": identity["path"],
        "phase": 6,
        "status": "frozen",
        "artifact_role": identity["artifact_role"],
        "producer": producer,
        "shape": (
            None
            if shape is None
            else [int(value) for value in shape]
        ),
        "size_bytes": int(identity["size_bytes"]),
        "sha256": identity["sha256"],
        "inputs": [
            {
                "type": "artifact",
                "artifact_id": input_artifact_id,
            }
            for input_artifact_id in artifact_inputs[artifact_id]
        ],
    }

existing_601_ids = sorted(
    set(registry_entries)
    & set(artifact_registry["artifacts"])
)

print("Registry entries prepared:", len(registry_entries))
print("Existing conflicting artifact IDs:", existing_601_ids)

print(
    json.dumps(
        registry_entries,
        indent=2,
        sort_keys=True,
    )
)

Registry entries prepared: 8
Existing conflicting artifact IDs: ['phase6.601.analysis_metadata', 'phase6.601.drug_level_results', 'phase6.601.lolo_performance', 'phase6.601.primary_cv_partitions', 'phase6.601.primary_repeat_performance']
{
  "phase6.601.analysis_metadata": {
    "artifact_role": "metadata",
    "inputs": [
      {
        "artifact_id": "phase6.601.primary_cv_partitions",
        "type": "artifact"
      },
      {
        "artifact_id": "phase6.601.primary_oof_predictions",
        "type": "artifact"
      },
      {
        "artifact_id": "phase6.601.primary_program_fold_parameters",
        "type": "artifact"
      },
      {
        "artifact_id": "phase6.601.primary_program_fold_lineage_effects",
        "type": "artifact"
      },
      {
        "artifact_id": "phase6.601.primary_repeat_performance",
        "type": "artifact"
      },
      {
        "artifact_id": "phase6.601.lolo_performance",
        "type": "artifact"
      },
      {
        "artifact_id":

In [35]:
# =============================================================================
# Validate notebook-601 additions against the full artifact registry
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    validate_artifact_registry,
)

updated_artifact_registry = json.loads(
    json.dumps(artifact_registry)
)

updated_artifact_registry["artifacts"].update(
    registry_entries
)

validated_artifact_registry = validate_artifact_registry(
    updated_artifact_registry
)

new_601_ids = sorted(
    artifact_id
    for artifact_id in validated_artifact_registry["artifacts"]
    if artifact_id.startswith("phase6.601.")
)

print(
    "Total registered artifacts:",
    len(validated_artifact_registry["artifacts"]),
)
print(
    "Notebook-601 artifacts:",
    len(new_601_ids),
)
print(
    "Registered notebook-601 IDs:",
)
for artifact_id in new_601_ids:
    print(f"- {artifact_id}")

Total registered artifacts: 117
Notebook-601 artifacts: 8
Registered notebook-601 IDs:
- phase6.601.analysis_metadata
- phase6.601.drug_level_results
- phase6.601.lolo_performance
- phase6.601.primary_cv_partitions
- phase6.601.primary_oof_predictions
- phase6.601.primary_program_fold_lineage_effects
- phase6.601.primary_program_fold_parameters
- phase6.601.primary_repeat_performance


In [36]:
# =============================================================================
# Persist notebook-601 artifact-registry additions
# =============================================================================

Paths.artifact_registry.write_text(
    json.dumps(
        validated_artifact_registry,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

artifact_registry = load_artifact_registry()

persisted_601_ids = sorted(
    artifact_id
    for artifact_id in artifact_registry["artifacts"]
    if artifact_id.startswith("phase6.601.")
)

print("Artifact registry:", project_relative_path(Paths.artifact_registry))
print(
    "Total registered artifacts:",
    len(artifact_registry["artifacts"]),
)
print(
    "Persisted notebook-601 artifacts:",
    len(persisted_601_ids),
)

for artifact_id in persisted_601_ids:
    print(f"- {artifact_id}")

Artifact registry: config/artifact_registry.json
Total registered artifacts: 117
Persisted notebook-601 artifacts: 8
- phase6.601.analysis_metadata
- phase6.601.drug_level_results
- phase6.601.lolo_performance
- phase6.601.primary_cv_partitions
- phase6.601.primary_oof_predictions
- phase6.601.primary_program_fold_lineage_effects
- phase6.601.primary_program_fold_parameters
- phase6.601.primary_repeat_performance


In [37]:
# =============================================================================
# Verify identities of newly registered notebook-601 artifacts
# =============================================================================

identity_findings = []

for artifact_id in persisted_601_ids:
    artifact = artifact_registry["artifacts"][artifact_id]
    path = Paths.root / artifact["path"]

    actual_size = path.stat().st_size
    actual_sha256 = calculate_sha256(path)

    if actual_size != artifact["size_bytes"]:
        identity_findings.append(
            {
                "artifact_id": artifact_id,
                "issue": "size_mismatch",
            }
        )

    if actual_sha256 != artifact["sha256"]:
        identity_findings.append(
            {
                "artifact_id": artifact_id,
                "issue": "sha256_mismatch",
            }
        )

print(
    "Notebook-601 registered artifacts verified:",
    len(persisted_601_ids),
)
print(
    "Identity findings:",
    identity_findings,
)

Notebook-601 registered artifacts verified: 8
Identity findings: []


## Conclusion and handoff

Notebook 601 evaluated whether the three frozen Phase 4 consensus transcriptomic programs provide reproducible predictive information about GDSC `LN_IC50` beyond a lineage-only baseline under the prospectively frozen evaluation design.

### Primary evaluation

All 281 prospectively eligible GDSC drugs were evaluated using `5-fold lineage-stratified cross-validation × 5 repeats` with deterministic seed `601`.

Across drugs:

* median lineage-only `R²` was `0.231`;
* median lineage-plus-program `R²` was `0.244`;
* median incremental predictive contribution was `ΔR² = 0.016`;
* 275/281 drugs satisfied the minimum program-model `R²` criterion;
* 126/281 satisfied the minimum `median ΔR²` criterion;
* 226/281 showed positive `ΔR²` in at least 4 of 5 repeats;
* 125/281 satisfied all prospectively frozen criteria for subsequent primary program-level SHAP interpretation.

The dominant reason for gate failure was insufficient incremental predictive contribution beyond lineage rather than inadequate overall predictive performance.

### Secondary unseen-lineage stress test

The prospectively defined leave-one-supported-lineage-out stress test comprised 2,961 `drug × held-out lineage` evaluations.

Across drugs:

* median unseen-lineage program-model `R²` was `-0.088`;
* median `ΔR²` relative to the training-derived intercept-only reference was `0.157`;
* the median fraction of held-out lineages with positive `ΔR²` was `0.727`.

These results indicate heterogeneous relative transportability of the frozen programs but limited absolute prediction when an entire lineage is absent during training.

The LOLO analysis remains secondary and descriptive. It does not modify primary eligibility for notebook 602 and does not provide a rescue or exclusion route.

### Stable notebook-601 handoffs

The following frozen artifacts were persisted and registered:

* `phase6.601.primary_cv_partitions`
* `phase6.601.primary_repeat_performance`
* `phase6.601.lolo_performance`
* `phase6.601.drug_level_results`
* `phase6.601.analysis_metadata`

All five registered artifact identities were verified locally after persistence.

### Analytical boundary

Notebook 601 provides internal developmental evidence about resistance-like pharmacogenomic contexts in GDSC.

It does not establish:

* clinical drug-resistance prediction;
* causal or mechanistic relationships;
* therapeutic efficacy;
* validated biomarkers;
* patient-level generalization; or
* external cross-screen reproducibility.

CTRP and PRISM remained sealed for predictive outcomes during notebook 601, and Phase 5 evidence was not used for feature selection, drug selection, model selection, threshold definition, or result rescue.

Notebook 602 should proceed only after its SHAP attribution rules are prospectively frozen.
